# finrag on Google Colab
Questions on company financial reports (10-K, 10-Q, 8-K, earnings), answered with retrieval + a 7B model and evaluated on the open FinanceBench sample.

Runtime → Change runtime type → **T4 GPU** before you start. Run the cells in order.

In [ ]:
!nvidia-smi | head -15

## 1. Install and test
Replace the URL with your repository. The tests need no GPU and no download: they must all pass.

In [ ]:
REPO_URL = "https://github.com/YOUR-USER/financial-reports-rag"  # <- change this
!git clone $REPO_URL finrag-repo
%cd finrag-repo
!pip install -q -e ".[ml,api,dev]"
!pytest

## 2. (Optional) Restore earlier work from Google Drive
Colab storage is wiped when the session ends. If you saved `data/` in a previous session, restore it here and skip step 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = "/content/drive/MyDrive/finrag"
!mkdir -p $SAVE && ls $SAVE
# !cp -r $SAVE/data .     # <- uncomment to restore

## 3. Download the reports and build the index
`fetch` downloads the 150 FinanceBench questions and the PDFs they refer to (about 84 reports, a few minutes).
Use `--max-docs 20` for a quicker first run. `ingest` reads every page, chunks it and embeds it on the GPU.

In [ ]:
!python -m finrag fetch
!python -m finrag ingest
!python -m finrag make-gold

## 4. Ask questions
The first call downloads the language model (several GB): **check its licence** first.
Read every answer critically: open the cited page in the PDF (`data/pdf/`) and check the number.

In [ ]:
!python -m finrag ask "What is the FY2018 capital expenditure amount (in USD millions) for 3M?" --doc 3M_2018_10K

In [ ]:
!python -m finrag ask "What is the FY2018 capital expenditure amount (in USD millions) for 3M?"

In [ ]:
!python -m finrag ask --mode agent "By how much did 3M's capital expenditure change between FY2017 and FY2018, in percent?"

## 5. Evaluation
`data/eval/gold.jsonl` holds the FinanceBench questions whose reports are indexed. **Add your ~8 unanswerable questions** to it
(see the README for the format). Each answer takes ~10-20 s on a T4: start with `--limit 40` and one or two variants.

In [ ]:
!python -m finrag eval --limit 40 --variant "hybrid, shared" --variant "hybrid, per_doc"

In [ ]:
!python -m finrag eval --mode agent --limit 20 --variant "hybrid, per_doc" --out data/eval/results_agent.json

## 6. Keep your work
Copy the index and the evaluation files to Google Drive (step 2 restores them next time).

In [ ]:
!mkdir -p $SAVE && cp -r data $SAVE/
!ls -R $SAVE/data/eval